In [10]:
import os
from pathlib import Path

# Get the directory of the notebook, then its parent (the project root)
# os.getcwd() is reliable in notebooks, but Path(__file__).resolve().parent 
# is preferred for scripts. We'll use a robust approach for the notebook.
PROJECT_ROOT = Path(os.getcwd()).parent

# Define directories based on the project structure
recruits_base_dir = PROJECT_ROOT / "data" / "Football Recruitment Tables"
scouting_reports_dir = recruits_base_dir / "Scouting Reports"
recruitment_classes_dir = recruits_base_dir / "Recruitment Classes"
output_dir = PROJECT_ROOT / "data" / "modeling_datasets" / "recruits"

print(f"Project root directory: {PROJECT_ROOT}")
print(f"Scouting Reports directory: {scouting_reports_dir}")
print(f"Recruitment Classes directory: {recruitment_classes_dir}")

# Ensure existence
for d in [scouting_reports_dir, recruitment_classes_dir, output_dir]:
    if not d.exists():
        print(f"Creating directory: {d}")
        d.mkdir(parents=True, exist_ok=True)
    else:
        print(f"Directory exists: {d}")


Project root directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence
Scouting Reports directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\Scouting Reports
Recruitment Classes directory: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\Recruitment Classes
Directory exists: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\Scouting Reports
Directory exists: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\Football Recruitment Tables\Recruitment Classes
Directory exists: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits


In [11]:
import pandas as pd
import numpy as np
import os
import re
import csv

def process_scouting_reports(start_year=2015, end_year=2028, input_dir=scouting_reports_dir, output_path=output_dir / 'master_scouting_reports_2015_2028.csv'):
    """
    Reads scouting report CSVs across multiple years, extracts athletic background flags,
    pivots skill categories into distinct columns, and merges them into one master file.
    
    Robustness Fixes:
    - Added lineterminator='\n' to ensure cross-platform compatibility.
    - Added quoting=csv.QUOTE_ALL to protect fields containing newlines.
    - Added encoding='utf-8-sig' for better Excel compatibility.
    """
    all_years_data = []
    excluded_skill_names = {"url", "hs_athletic_background", "hs athletic background", "skill_url", "skill_hs_athletic_background"}
    
    for year in range(start_year, end_year + 1):
        file_name = input_dir / f'scouting_reports_{year}.csv'
        
        if not file_name.exists():
            if year < 2026:
                print(f"⚠️ Skipping {year} (File not found: {file_name})")
            continue
            
        print(f"🏈 Processing data for {year}...")
        # Read with high-fidelity settings to catch embedded newlines correctly
        df = pd.read_csv(file_name)
        
        # Clean up any potential \r characters that might have snuck in from Windows files
        if 'hs_athletic_background' in df.columns:
            df['hs_athletic_background'] = df['hs_athletic_background'].fillna('').str.replace('\r', '', regex=False)
        
        # 1. ADD BACKGROUND FLAGS (Regex matching)
        bg = df['hs_athletic_background'].str.lower()
        
        df['flag_all_usa_first_team'] = bg.str.contains(r'all-usa first team|usa today first team').astype(int)
        df['flag_all_usa_second_team'] = bg.str.contains(r'all-usa second team|usa today second team').astype(int)
        df['flag_all_american_bowl'] = bg.str.contains(r'all-american bowl|army all-american').astype(int)
        df['flag_all_america_game'] = bg.str.contains(r'under armour all-america|all-america game').astype(int)
        df['flag_gatorade_poy'] = bg.str.contains(r'gatorade player of the year|gatorade.*poy').astype(int)
        df['flag_state_poy'] = bg.str.contains(r'state player of the year').astype(int)
        df['flag_the_opening'] = bg.str.contains(r'the opening').astype(int)
        df['flag_maxpreps_all_american'] = bg.str.contains(r'maxpreps.*all-american').astype(int)
        
        # 2. FLATTEN & PIVOT SKILLS
        skills_data = []
        for _, row in df.iterrows():
            player_id = row['player_id']
            player_skills = {'player_id': player_id}
            
            for i in range(1, 9):
                skill_col = f'skill_{i}'
                rating_col = f'skill_{i}_rating'
                
                if skill_col in df.columns and pd.notna(row.get(skill_col)):
                    skill_name = str(row[skill_col]).strip()
                    if skill_name:
                        skill_name_norm = skill_name.lower().strip().replace('-', ' ').replace('_', ' ')
                        if skill_name_norm in {"url", "hs athletic background"}:
                            continue
                        prefixed_skill_name = skill_name if skill_name.startswith('skill_') else f"skill_{skill_name}"
                        if prefixed_skill_name.lower().strip() in excluded_skill_names:
                            continue
                        player_skills[prefixed_skill_name] = row.get(rating_col, np.nan)
            
            skills_data.append(player_skills)
            
        skills_df = pd.DataFrame(skills_data)
        
        # 3. CLEAN UP & MERGE
        cols_to_drop = [f'skill_{i}' for i in range(1, 9)] + [f'skill_{i}_rating' for i in range(1, 9)]
        df_clean = df.drop(columns=[col for col in cols_to_drop if col in df.columns], errors='ignore')
        
        final_yearly_df = df_clean.merge(skills_df, on='player_id', how='left')
        all_years_data.append(final_yearly_df)

    # 4. COMBINE ALL YEARS
    if all_years_data:
        print("\n🔄 Combining all years into a single master dataset...")
        master_df = pd.concat(all_years_data, ignore_index=True)
        
        # Final cleanup for the master file - strip out \r to ensure \n only
        for col in master_df.select_dtypes(include=['object']):
            master_df[col] = master_df[col].fillna('').astype(str).str.replace('\r', '', regex=False)
        
        # Ensure output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save to CSV using robust formatting for long text fields
        master_df.to_csv(
            output_path, 
            index=False, 
            encoding='utf-8-sig',      # Excel friendly
            quoting=csv.QUOTE_ALL,     # Protects fields with newlines/commas
            lineterminator='\n'        # Clean line endings
        )
        
        print(f"✅ Success! Master file saved as: {output_path}")
        print(f"📊 Total Players Processed: {master_df.shape[0]}")
        print(f"🎯 Total Unique Skills Found: {len(master_df.columns) - len(df_clean.columns)}")
        
        return master_df
    else:
        print("❌ No files were processed.")
        return None

# Execute the function
if __name__ == "__main__":
    df_master = process_scouting_reports()
    if df_master is not None:
        display(df_master.head())


🏈 Processing data for 2015...
🏈 Processing data for 2016...
🏈 Processing data for 2017...
🏈 Processing data for 2018...
🏈 Processing data for 2019...
🏈 Processing data for 2020...
🏈 Processing data for 2021...
🏈 Processing data for 2022...
🏈 Processing data for 2023...
🏈 Processing data for 2024...
🏈 Processing data for 2025...
🏈 Processing data for 2026...
🏈 Processing data for 2027...
🏈 Processing data for 2028...

🔄 Combining all years into a single master dataset...
✅ Success! Master file saved as: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\master_scouting_reports_2015_2028.csv
📊 Total Players Processed: 41925
🎯 Total Unique Skills Found: 69


,player_id,hs_athletic_background,flag_all_usa_first_team,flag_all_usa_second_team,flag_all_american_bowl,flag_all_america_game,flag_gatorade_poy,flag_state_poy,flag_the_opening,flag_maxpreps_all_american,...,skill_Pass-rush ability,skill_Playmaking ability,skill_Second Level Blocking,skill_Leadership,skill_Scorer/Finisher,skill_Passing/Vision,skill_Defender,skill_Passing,skill_Penetration Ability,skill_Handle
0,201500001,2015 U.S. Army All-American Bowl selection.......,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201500002,2015 U.S. Army All-American Bowl selection. .....,1,0,1,0,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201500003,Named 2014 All-USA First Team by USA TODAY...W...,1,0,0,0,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,201500004,Named 2014 All-USA First Team by USA TODAY...2...,1,0,0,1,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,201500005,PERSONAL: Consensus top defensive back in the ...,0,0,0,1,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
def process_recruitment_classes(start_year=2015, end_year=2028, input_dir=recruitment_classes_dir, output_path=output_dir / 'master_recruits_2015_2028.csv'):
    """
    Reads individual recruitment class CSVs across multiple years and combines them into one master file.
    
    Robustness Fixes:
    - Added lineterminator='\n' to ensure cross-platform compatibility.
    - Added quoting=csv.QUOTE_ALL to protect fields containing commas or quotes.
    - Added encoding='utf-8-sig' for better Excel compatibility.
    """
    all_years_recruits = []
    
    for year in range(start_year, end_year + 1):
        file_name = input_dir / f'recruits_{year}.csv'
        
        if not file_name.exists():
            if year < 2026:
                print(f"⚠️ Skipping recruitment {year} (File not found: {file_name})")
            continue
            
        print(f"🎓 Aggregating recruits for {year}...")
        try:
            # Read recruitment data
            df = pd.read_csv(file_name)
            
            # Basic validation/cleanup
            if 'player_id' not in df.columns:
                print(f"⚠️ Warning: 'player_id' not found in {file_name}")
            
            all_years_recruits.append(df)
        except Exception as e:
            print(f"❌ Error reading {file_name}: {e}")

    # Combine all years
    if all_years_recruits:
        print("\n🔄 Combining all recruitment classes into a single master file...")
        master_recruits_df = pd.concat(all_years_recruits, ignore_index=True)
        
        # Ensure output directory exists
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save to CSV using robust formatting
        master_recruits_df.to_csv(
            output_path, 
            index=False, 
            encoding='utf-8-sig',      # Excel friendly
            quoting=csv.QUOTE_ALL,     # Protects fields with commas/quotes
            lineterminator='\n'        # Clean line endings
        )
        
        print(f"✅ Success! Master recruits file saved as: {output_path}")
        print(f"📊 Total Records Combined: {master_recruits_df.shape[0]}")
        
        return master_recruits_df
    else:
        print("❌ No recruitment files were processed.")
        return None

# Execute the function
if __name__ == "__main__":
    df_recruits = process_recruitment_classes()
    if df_recruits is not None:
        display(df_recruits.head())

🎓 Aggregating recruits for 2015...
🎓 Aggregating recruits for 2016...
🎓 Aggregating recruits for 2017...
🎓 Aggregating recruits for 2018...
🎓 Aggregating recruits for 2019...
🎓 Aggregating recruits for 2020...
🎓 Aggregating recruits for 2021...
🎓 Aggregating recruits for 2022...
🎓 Aggregating recruits for 2023...
🎓 Aggregating recruits for 2024...
🎓 Aggregating recruits for 2025...
🎓 Aggregating recruits for 2026...
🎓 Aggregating recruits for 2027...
🎓 Aggregating recruits for 2028...

🔄 Combining all recruitment classes into a single master file...
✅ Success! Master recruits file saved as: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\master_recruits_2015_2028.csv
📊 Total Records Combined: 41926


,player_id,year,rank,name,position,height,weight,rating,high_school,city,state,committed_to,url
0,201500001,2015,1,Trenton Thompson,DT,"6' 2.5""",313 lbs,0.9991,Westover,Albany,GA,Georgia,https://247sports.com/player/trenton-thompson-...
1,201500002,2015,2,Martez Ivey,OT,"6' 5.5""",275 lbs,0.9991,Apopka,Apopka,FL,Florida,https://247sports.com/player/martez-ivey-26822/
2,201500003,2015,3,Byron Cowart,SDE,"6' 4""",250 lbs,0.9987,Armwood,Seffner,FL,Auburn,https://247sports.com/player/byron-cowart-23209/
3,201500004,2015,4,Iman Marshall,CB,"6' 1""",190 lbs,0.9985,Long Beach Poly,Long Beach,CA,USC,https://247sports.com/player/iman-marshall-19987/
4,201500005,2015,5,Derwin James,S,"6' 2""",201 lbs,0.9982,Haines City,Auburndale,FL,Florida State,https://247sports.com/player/derwin-james-19186/


In [13]:
# --- Final Merge: Recruitment Classes + Scouting Reports ---

def combine_all_recruiting_data(df_scouts, df_recruits, output_path=output_dir / 'modeling_master_recruiting.csv'):
    """
    Final step: Merges the aggregated recruitment data (rank/rating) with scouting reports (flags/skills).
    Uses a 'left' join on recruits to ensure all ranked players are kept.
    """
    if df_scouts is None or df_recruits is None:
        print("❌ Error: One or both dataframes are missing.")
        return None
    
    print(f"📊 Merging {len(df_recruits)} recruits with {len(df_scouts)} scouting reports...")
    
    # Merge on player_id
    # We use left join on df_recruits because that's our 'ground truth' roster list.
    final_df = df_recruits.merge(df_scouts, on='player_id', how='left', suffixes=('', '_scout'))
    
    # CLEANUP: Handle potential duplicate columns after merge (e.g., player names if they existed in both)
    # Most of the time, recruits file has the 'canonical' name.
    if 'name_scout' in final_df.columns:
        final_df = final_df.drop(columns=['name_scout'])
    
    # Save the final dataset
    final_df.to_csv(
        output_path, 
        index=False, 
        encoding='utf-8-sig', 
        quoting=csv.QUOTE_ALL, 
        lineterminator='\n'
    )
    
    print(f"✅ FINAL SUCCESS! Merged dataset saved as: {output_path}")
    print(f"📊 Final Dataset Shape: {final_df.shape}")
    
    return final_df

if __name__ == "__main__":
    # Note: df_master comes from the scouting report aggregation, 
    # and df_recruits comes from the recruitment classes aggregation.
    df_final_modeling = combine_all_recruiting_data(df_master, df_recruits)
    if df_final_modeling is not None:
        display(df_final_modeling.head())

📊 Merging 41926 recruits with 41925 scouting reports...
✅ FINAL SUCCESS! Merged dataset saved as: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\modeling_master_recruiting.csv
📊 Final Dataset Shape: (41926, 91)


,player_id,year,rank,name,position,height,weight,rating,high_school,city,...,skill_Pass-rush ability,skill_Playmaking ability,skill_Second Level Blocking,skill_Leadership,skill_Scorer/Finisher,skill_Passing/Vision,skill_Defender,skill_Passing,skill_Penetration Ability,skill_Handle
0,201500001,2015,1,Trenton Thompson,DT,"6' 2.5""",313 lbs,0.9991,Westover,Albany,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,201500002,2015,2,Martez Ivey,OT,"6' 5.5""",275 lbs,0.9991,Apopka,Apopka,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,201500003,2015,3,Byron Cowart,SDE,"6' 4""",250 lbs,0.9987,Armwood,Seffner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,201500004,2015,4,Iman Marshall,CB,"6' 1""",190 lbs,0.9985,Long Beach Poly,Long Beach,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,201500005,2015,5,Derwin James,S,"6' 2""",201 lbs,0.9982,Haines City,Auburndale,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# --- Unique Position Counts ---

print("📊 Final Merged Dataset Position Counts:")
if 'position' in df_final_modeling.columns:
    position_counts = df_final_modeling['position'].value_counts()
    print(position_counts)
else:
    print("❌ 'position' column not found in df_final_modeling.")


📊 Final Merged Dataset Position Counts:
position
WR      5380
CB      3621
ATH     3381
OT      3326
S       3182
RB      2816
TE      1942
DL      1855
LB      1732
OLB     1598
IOL     1564
DT      1423
OG      1362
Edge    1245
QB      1245
ILB     1120
SDE     1081
WDE     1064
PRO      953
DUAL     588
K        374
OC       355
APB      303
P        216
LS       137
FB        62
RET        1
Name: count, dtype: int64


In [15]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# --- CONFIGURATION ---
INPUT_FILE = output_dir / "modeling_master_recruiting.csv"
OUTPUT_DIR = output_dir / "positional"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. POSITIONAL MAPPING
POS_MAP = {
    'PRO': 'QB', 'DUAL': 'QB', 'QB': 'QB',
    'RB': 'RB', 'APB': 'RB', 'FB': 'RB',
    'WR': 'WR',
    'TE': 'TE',
    'OT': 'OL', 'OG': 'OL', 'OC': 'OL', 'IOL': 'OL',
    'DT': 'IDL', 'DL': 'IDL',
    'SDE': 'EDGE', 'WDE': 'EDGE', 'Edge': 'EDGE',
    'ILB': 'LB', 'OLB': 'LB', 'LB': 'LB',
    'CB': 'DB', 'S': 'DB',
    'ATH': 'ATH',
    'K': 'SPEC', 'P': 'SPEC', 'LS': 'SPEC', 'RET': 'SPEC'
}

SKILL_EXCLUDE_COLS = {'url', 'hs_athletic_background', 'skill_url', 'skill_hs_athletic_background'}

# --- CLEANING FUNCTIONS ---
def clean_height(h):
    """Converts '6' 2.5"' or '6-2' into total inches (e.g., 74.5)"""
    if pd.isna(h):
        return np.nan
    h_str = str(h).strip()
    # Match formats like 6' 2" or 6' 2.5"
    m1 = re.search(r"(\d+)['\-]\s*([\d\.]+)%?\"?", h_str)
    if m1:
        return (int(m1.group(1)) * 12) + float(m1.group(2))
    # Match format like 6'
    m2 = re.search(r"(\d+)['\-]", h_str)
    if m2:
        return int(m2.group(1)) * 12
    return np.nan

def clean_weight(w):
    """Converts '170 lbs' into 170.0"""
    if pd.isna(w):
        return np.nan
    m = re.search(r"(\d+)", str(w))
    if m:
        return float(m.group(1))
    return np.nan


def process_positional_files():
    print(f"Loading master dataset...")
    df = pd.read_csv(INPUT_FILE, low_memory=False)
    
    # 2. APPLY CLEANING
    print("Cleaning Heights and Weights...")
    df['height_inches'] = df['height'].apply(clean_height)
    df['weight_lbs'] = df['weight'].apply(clean_weight)
    
    # We can drop the original uncleaned columns to keep things tidy
    df = df.drop(columns=['height', 'weight'])
    
    # Convert rating safely and fill unrated players with 0.75
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce').fillna(0.75)
    df.loc[df['rating'] == 0.0, 'rating'] = 0.75
    
    # Map positions
    df['pos_group'] = df['position'].map(POS_MAP)
    
    # 3. IDENTIFY METADATA VS SKILL COLUMNS
    metadata_cols = [
        'player_id', 'year', 'rank', 'name', 'position', 'height_inches', 'weight_lbs', 
        'rating', 'high_school', 'city', 'state', 'committed_to', 'pos_group',
        'url', 'hs_athletic_background'
    ]
    flag_cols = [c for c in df.columns if c.startswith('flag_')]
    non_skill_cols = set(metadata_cols + flag_cols + list(SKILL_EXCLUDE_COLS))
    
    # Everything else is a Skill column
    skill_cols = [c for c in df.columns if c not in non_skill_cols]

    # Prefix skill columns for modeling readability (idempotent: avoids double-prefixing)
    skill_rename_map = {
        col: f"skill_{col}"
        for col in skill_cols
        if not col.startswith("skill_") and col not in SKILL_EXCLUDE_COLS
    }
    if skill_rename_map:
        df = df.rename(columns=skill_rename_map)
        print(f"Prefixed {len(skill_rename_map)} skill columns with 'skill_'.")

    # Recompute skill columns after renaming
    flag_cols = [c for c in df.columns if c.startswith('flag_')]
    non_skill_cols = set(metadata_cols + flag_cols + list(SKILL_EXCLUDE_COLS))
    skill_cols = [
        c for c in df.columns
        if c not in non_skill_cols and c.startswith('skill_') and c not in SKILL_EXCLUDE_COLS
    ]

    # 4. PROCESS EACH POSITION GROUP
    for group, group_df in df.groupby('pos_group'):
        print(f"\n🏈 Processing {group} ({len(group_df)} players)...")
        group_df = group_df.copy()
        
        # A. Find the Top 10 skills (highest non-null counts)
        non_null_counts = group_df[skill_cols].notna().sum()
        
        # Keep the top 10 that actually have at least 1 piece of data
        top_10_skills = non_null_counts[non_null_counts > 0].nlargest(10).index.tolist()
        
        cols_to_drop = [col for col in skill_cols if col not in top_10_skills]
        group_df = group_df.drop(columns=cols_to_drop)
        print(f"   -> Kept Top 10 Skills: {', '.join(top_10_skills)}")
        
        # B. Impute missing Top 10 skills
        for col in top_10_skills:
            group_df[col] = pd.to_numeric(group_df[col], errors='coerce')
            
            # Global fallback array for this skill
            global_array = group_df[col].dropna().values
            
            for year, year_df in group_df.groupby('year'):
                year_array = year_df[col].dropna().values
                
                # If a specific year has less than 5 data points, fallback to the global distribution
                dist_array = year_array if len(year_array) >= 5 else global_array
                
                # Find players missing this skill
                missing_idx = year_df[year_df[col].isna()].index
                
                if len(missing_idx) > 0 and len(dist_array) > 0:
                    ratings = group_df.loc[missing_idx, 'rating'].astype(float).values
                    
                    # MATH: Map rating [0.70, 0.99] to quantile [0.01, 0.90]
                    target_quantiles = np.interp(ratings, [0.70, 0.99], [0.01, 0.90])
                    
                    # Use np.nanquantile to safely pull values from the distribution based on the calculated quantiles
                    imputed_vals = np.nanquantile(dist_array, target_quantiles)
                    
                    # Assign back
                    group_df.loc[missing_idx, col] = np.round(imputed_vals, 2)
                    
        # C. Save the Positional Subfile
        out_path = OUTPUT_DIR / f"{group}_modeling.csv"
        group_df.to_csv(out_path, index=False)
        print(f"   -> Saved to {out_path.name}")

if __name__ == "__main__":
    process_positional_files()
    print("\n✅ All positional modeling files generated successfully!")

Loading master dataset...
Cleaning Heights and Weights...

🏈 Processing ATH (3381 players)...
   -> Kept Top 10 Skills: skill_Size, skill_Speed, skill_Change of Direction, skill_Explosiveness, skill_Athleticism, skill_Ball Skills, skill_Intangibles, skill_Versatility, skill_Feet, skill_Frame
   -> Saved to ATH_modeling.csv

🏈 Processing DB (6803 players)...
   -> Kept Top 10 Skills: skill_Ball Skills, skill_Speed, skill_Instincts, skill_Tackling, skill_Change of Direction, skill_Size, skill_Length, skill_Recovery Speed, skill_Reactive Quickness, skill_Frame
   -> Saved to DB_modeling.csv

🏈 Processing EDGE (3390 players)...
   -> Kept Top 10 Skills: skill_Athleticism, skill_Hand Quickness, skill_Point of Attack, skill_First Step, skill_Frame, skill_Closing Speed, skill_Size, skill_Pursuit, skill_Speed, skill_Instincts
   -> Saved to EDGE_modeling.csv

🏈 Processing IDL (3278 players)...
   -> Kept Top 10 Skills: skill_Athleticism, skill_Hand Quickness, skill_Point of Attack, skill_Size,

In [16]:


import pandas as pd
import numpy as np
import difflib
from pathlib import Path

# --- CONFIGURATION ---
FILE_MODELING = output_dir / "modeling_master_recruiting.csv"
FILE_CFBD = PROJECT_ROOT / "data" / "Football Recruitment Tables" / "cfbd_recruits" / "recruiting_summaries_2015_2028.csv"
OUTPUT_FILE = output_dir / "recruit_player_id_mapping_bridge.csv"

# Minimum fuzzy match ratio (0.0 to 1.0) to consider it a match
NAME_SIMILARITY_THRESHOLD = 0.85 

POS_MAP = {
    'PRO': 'QB', 'DUAL': 'QB', 'QB': 'QB',
    'RB': 'RB', 'APB': 'RB', 'FB': 'RB',
    'WR': 'WR',
    'TE': 'TE',
    'OT': 'OL', 'OG': 'OL', 'OC': 'OL', 'IOL': 'OL',
    'DT': 'IDL', 'DL': 'IDL',
    'SDE': 'EDGE', 'WDE': 'EDGE', 'Edge': 'EDGE',
    'ILB': 'LB', 'OLB': 'LB', 'LB': 'LB',
    'CB': 'DB', 'S': 'DB',
    'ATH': 'ATH',
    'K': 'SPEC', 'P': 'SPEC', 'LS': 'SPEC', 'RET': 'SPEC'
}

def string_similarity(s1, s2):
    """Calculates string similarity ratio using difflib."""
    if pd.isna(s1) or pd.isna(s2):
        return 0.0
    return difflib.SequenceMatcher(None, str(s1).lower().strip(), str(s2).lower().strip()).ratio()

def is_rating_match(r1, r2, threshold=0.05):
    """Checks if ratings are within a 5% delta of each other."""
    if pd.isna(r1) or pd.isna(r2) or r1 == 0 or r2 == 0:
        return False
    # Check absolute percentage difference relative to the larger rating
    return abs(r1 - r2) / max(r1, r2) <= threshold

def create_player_bridge():
    print("Loading datasets...")
    df_mod = pd.read_csv(FILE_MODELING, low_memory=False)
    df_cfbd = pd.read_csv(FILE_CFBD, low_memory=False)

    # 1. STANDARDIZE COLUMNS FOR MATCHING
    df_mod['pos_group'] = df_mod['position'].map(POS_MAP).fillna('UNKNOWN')
    df_cfbd['pos_group'] = df_cfbd['position'].map(POS_MAP).fillna('UNKNOWN')

    # Ensure ratings are numeric and filled
    df_mod['rating'] = pd.to_numeric(df_mod['rating'], errors='coerce').fillna(0.75)
    df_cfbd['rating'] = pd.to_numeric(df_cfbd['rating'], errors='coerce').fillna(0.75)
    
    df_mod['state'] = df_mod['state'].str.strip().str.upper()
    df_cfbd['state'] = df_cfbd['state'].str.strip().str.upper()

    # 2. RENAME COLUMNS FOR CLARITY IN THE BRIDGE
    # We prefix to keep track of where the data came from
    df_mod = df_mod[['player_id', 'year', 'pos_group', 'state', 'name', 'position', 'rating']].rename(
        columns={'name': 'name_mod', 'position': 'pos_mod', 'rating': 'rating_mod'}
    )
    
    df_cfbd = df_cfbd[['id', 'athlete_id', 'year', 'pos_group', 'state', 'name', 'position', 'rating']].rename(
        columns={'id': 'cfbd_recruit_id', 'name': 'name_cfbd', 'position': 'pos_cfbd', 'rating': 'rating_cfbd'}
    )

    matched_records = []
    matched_mod_ids = set()
    matched_cfbd_ids = set()

    # =========================================================
    # LAYER 1: Match on [Year, Pos Group, State]
    # =========================================================
    print("Running Layer 1 Matching (Year, Pos Group, State)...")
    
    # Create groups
    mod_groups_l1 = df_mod.groupby(['year', 'pos_group', 'state'])
    cfbd_groups_l1 = df_cfbd.groupby(['year', 'pos_group', 'state'])

    for key, mod_subset in mod_groups_l1:
        if key in cfbd_groups_l1.groups:
            cfbd_subset = cfbd_groups_l1.get_group(key)
            
            # Cartesian product within this specific cohort
            for _, m_row in mod_subset.iterrows():
                if m_row['player_id'] in matched_mod_ids:
                    continue
                    
                best_match = None
                best_score = NAME_SIMILARITY_THRESHOLD
                
                for _, c_row in cfbd_subset.iterrows():
                    if c_row['cfbd_recruit_id'] in matched_cfbd_ids:
                        continue
                        
                    # Check 5% rating constraint FIRST (faster than fuzzy matching)
                    if is_rating_match(m_row['rating_mod'], c_row['rating_cfbd']):
                        # Fuzzy match the names
                        sim_score = string_similarity(m_row['name_mod'], c_row['name_cfbd'])
                        
                        if sim_score >= best_score:
                            best_score = sim_score
                            best_match = c_row
                            
                if best_match is not None:
                    matched_mod_ids.add(m_row['player_id'])
                    matched_cfbd_ids.add(best_match['cfbd_recruit_id'])
                    
                    matched_records.append({
                        'player_id': m_row['player_id'],
                        'cfbd_recruit_id': best_match['cfbd_recruit_id'],
                        'athlete_id': best_match['athlete_id'],
                        'year': key[0],
                        'state': key[2],
                        'name_mod': m_row['name_mod'],
                        'name_cfbd': best_match['name_cfbd'],
                        'pos_mod': m_row['pos_mod'],
                        'pos_cfbd': best_match['pos_cfbd'],
                        'rating_mod': m_row['rating_mod'],
                        'rating_cfbd': best_match['rating_cfbd'],
                        'rating_delta': round(abs(m_row['rating_mod'] - best_match['rating_cfbd']), 4),
                        'match_layer': 'Layer 1 (Pos Group)'
                    })

    print(f"Layer 1 Matches Found: {len(matched_records)}")

    # =========================================================
    # LAYER 2: Fallback Match on [Year, State]
    # =========================================================
    print("Running Layer 2 Fallback (Year, State)...")
    
    # Filter out players that are already matched
    df_mod_unmatched = df_mod[~df_mod['player_id'].isin(matched_mod_ids)]
    df_cfbd_unmatched = df_cfbd[~df_cfbd['cfbd_recruit_id'].isin(matched_cfbd_ids)]
    
    mod_groups_l2 = df_mod_unmatched.groupby(['year', 'state'])
    cfbd_groups_l2 = df_cfbd_unmatched.groupby(['year', 'state'])

    l2_count = 0
    for key, mod_subset in mod_groups_l2:
        if key in cfbd_groups_l2.groups:
            cfbd_subset = cfbd_groups_l2.get_group(key)
            
            for _, m_row in mod_subset.iterrows():
                if m_row['player_id'] in matched_mod_ids:
                    continue
                    
                best_match = None
                best_score = NAME_SIMILARITY_THRESHOLD
                
                for _, c_row in cfbd_subset.iterrows():
                    if c_row['cfbd_recruit_id'] in matched_cfbd_ids:
                        continue
                        
                    if is_rating_match(m_row['rating_mod'], c_row['rating_cfbd']):
                        sim_score = string_similarity(m_row['name_mod'], c_row['name_cfbd'])
                        
                        if sim_score >= best_score:
                            best_score = sim_score
                            best_match = c_row
                            
                if best_match is not None:
                    matched_mod_ids.add(m_row['player_id'])
                    matched_cfbd_ids.add(best_match['cfbd_recruit_id'])
                    l2_count += 1
                    
                    matched_records.append({
                        'player_id': m_row['player_id'],
                        'cfbd_recruit_id': best_match['cfbd_recruit_id'],
                        'athlete_id': best_match['athlete_id'],
                        'year': key[0],
                        'state': key[1],
                        'name_mod': m_row['name_mod'],
                        'name_cfbd': best_match['name_cfbd'],
                        'pos_mod': m_row['pos_mod'],
                        'pos_cfbd': best_match['pos_cfbd'],
                        'rating_mod': m_row['rating_mod'],
                        'rating_cfbd': best_match['rating_cfbd'],
                        'rating_delta': round(abs(m_row['rating_mod'] - best_match['rating_cfbd']), 4),
                        'match_layer': 'Layer 2 (State Only)'
                    })

    print(f"Layer 2 Matches Found: {l2_count}")
    print(f"Total Matches: {len(matched_records)}")

    # 3. SAVE RESULTS
    if matched_records:
        df_bridge = pd.DataFrame(matched_records)
        df_bridge.to_csv(OUTPUT_FILE, index=False)
        print(f"✅ Bridge file successfully saved to {OUTPUT_FILE}")
    else:
        print("⚠️ No matches found.")

if __name__ == "__main__":
    create_player_bridge()

Loading datasets...
Running Layer 1 Matching (Year, Pos Group, State)...
Layer 1 Matches Found: 36971
Running Layer 2 Fallback (Year, State)...
Layer 2 Matches Found: 977
Total Matches: 37948
✅ Bridge file successfully saved to x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\recruit_player_id_mapping_bridge.csv


In [17]:
# --- REVISED: IDENTIFY HIGHEST RANKED UNMATCHED PLAYERS ---
import pandas as pd
import numpy as np
from pathlib import Path

# Load the master dataset
master_path = PROJECT_ROOT/ "data" / "modeling_datasets" / "recruits" / "modeling_master_recruiting.csv"
df = pd.read_csv(master_path, low_memory=False)

# A more robust way to find unmatched players:
# Check if many of the flag/skill columns are all null for a row.
# We'll pick a few characteristic scout columns.
scout_indicators = ['hs_athletic_background', 'flag_gatorade_poy', 'flag_state_poy', 'skill_Explosiveness']

# Convert columns to numeric/string as needed to check for nulls consistent with the merge output
def is_missing(row):
    # A player is unmatched if they have no athletic background text AND no Gatorade flag info
    # (Since flags are initialized to 0.0 or NaN, we check for NaN)
    # The merge results in NaN for missing right-side values.
    return pd.isna(row['hs_athletic_background']) or str(row['hs_athletic_background']).strip() == ''

unmatched_mask = df.apply(is_missing, axis=1)
df_unmatched = df[unmatched_mask].copy()

# Ensure numeric types for sorting
df_unmatched['rating'] = pd.to_numeric(df_unmatched['rating'], errors='coerce')

# Get Top 10 per year by rating (descending)
top_10_unmatched = df_unmatched.sort_values(['year', 'rating'], ascending=[True, False]).groupby('year').head(10)

# Select relevant columns for the report
report_cols = ['year', 'rank', 'name', 'position', 'rating', 'high_school', 'city', 'state', 'committed_to']
top_10_report = top_10_unmatched[report_cols]

# Save to CSV
output_csv = PROJECT_ROOT/ "data" / "modeling_datasets" / "recruits" / "top_10_unmatched_by_year.csv"
top_10_report.to_csv(output_csv, index=False, encoding='utf-8-sig')

print(f"📊 Summary:")
print(f"   Total Players: {len(df)}")
print(f"   Unmatched Found: {len(df_unmatched)}")
print(f"   Saved Report to: {output_csv}")

display(top_10_report.head(10))


📊 Summary:
   Total Players: 41926
   Unmatched Found: 2
   Saved Report to: x:\My Files\Courses\DSBA 6010 - LLM\Project\Code\Gridiron_Intelligence\data\modeling_datasets\recruits\top_10_unmatched_by_year.csv


,year,rank,name,position,rating,high_school,city,state,committed_to
17553,2019,2086,Trey Taylor,S,0.8167,Frisco Lone Star,Frisco,TX,Air Force
37921,2026,145,Jaelen Waters,CB,0.9393,Armwood,Seffner,FL,Miami
